# Weights & Biases: Experiment Tracking, Visualization, and Team Collaboration

## What Is W&B?

Imagine you are doing a science fair project and you need to show your teacher every experiment you ran,  
every measurement you took, every graph, and every conclusion — all in one beautiful report.  
Weights & Biases (W&B / wandb) does exactly this for ML teams.

**Weights & Biases** is an MLOps platform for:
- **Experiment tracking**: log metrics, hyperparameters, and model outputs
- **Visualization**: interactive dashboards, training curves, confusion matrices, sample predictions
- **Sweeps**: automated hyperparameter search (Bayesian, grid, random)
- **Artifacts**: version datasets, models, and reports
- **Reports**: shareable, collaborative ML notebooks

## Resources

- **Docs**: [https://docs.wandb.ai/](https://docs.wandb.ai/)
- **GitHub**: [https://github.com/wandb/wandb](https://github.com/wandb/wandb)
- **YouTube — W&B Tutorial**: [https://www.youtube.com/watch?v=krZMFNSFiTE](https://www.youtube.com/watch?v=krZMFNSFiTE)
- **YouTube — W&B Sweeps**: [https://www.youtube.com/watch?v=9zrmUIlScdY](https://www.youtube.com/watch?v=9zrmUIlScdY)
- **Free course**: [https://www.wandb.courses/](https://www.wandb.courses/)

## Installation

```bash
pip install wandb
```

**No API key needed for this notebook** — we use offline mode:
```bash
export WANDB_MODE=offline   # saves locally, syncs later with `wandb sync`
```

For team use, create a free account at [wandb.ai](https://wandb.ai) and run `wandb login`.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

# Use offline mode — no API key needed
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_SILENT"] = "true"  # suppress verbose output

try:
    import wandb
    WANDB_AVAILABLE = True
    print(f"wandb version: {wandb.__version__}")
    print("Running in OFFLINE mode — logs saved locally")
except ImportError:
    WANDB_AVAILABLE = False
    print("wandb not installed — simulated output shown. Install: pip install wandb")

# Generate dataset
np.random.seed(42)
X, y = make_classification(n_samples=3000, n_features=20, n_informative=12, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")

## Core Concept 1: wandb.init() — Starting a Run

Every experiment begins with `wandb.init()`. This creates a **Run** — a record of one experiment.

Key parameters:
- `project`: which project this run belongs to (like a folder)
- `name`: human-readable name for this run
- `config`: dict of hyperparameters (visible in dashboard)
- `tags`: list of labels for filtering runs
- `notes`: free-text description
- `mode`: 'online', 'offline', 'disabled'

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Simple neural network for classification
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)]
        for _ in range(num_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)]
        layers += [nn.Linear(hidden_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_with_wandb(config):
    """Train a model and log everything to W&B."""

    if WANDB_AVAILABLE:
        run = wandb.init(
            project="mlp-classification",
            name=f"mlp_h{config['hidden_dim']}_l{config['num_layers']}",
            config=config,          # all hyperparams captured here
            tags=["mlp", "synthetic"],
            notes="MLP trained on synthetic classification dataset",
        )
        cfg = wandb.config  # access config through wandb (allows sweeps to override)
    else:
        cfg = type('cfg', (), config)()  # simple namespace
        print(f"[Simulated W&B] wandb.init(project='mlp-classification', config={config})")

    # Build model from config
    X_t  = torch.tensor(X_train, dtype=torch.float32)
    y_t  = torch.tensor(y_train, dtype=torch.float32)
    X_te = torch.tensor(X_test,  dtype=torch.float32)
    y_te = torch.tensor(y_test,  dtype=torch.float32)

    model = MLP(
        input_dim  = X_train.shape[1],
        hidden_dim = cfg.hidden_dim,
        num_layers = cfg.num_layers,
        dropout    = cfg.dropout,
    )

    if WANDB_AVAILABLE:
        wandb.watch(model, log="all", log_freq=50)  # log gradients and weights

    optimizer = optim.Adam(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    best_val_acc = 0
    for epoch in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(X_t)
        loss = criterion(logits, y_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_te)
            val_loss   = criterion(val_logits, y_te).item()
            val_preds  = (val_logits > 0).float().numpy()
            val_acc    = accuracy_score(y_test, val_preds)
            val_auc    = roc_auc_score(y_test, torch.sigmoid(val_logits).numpy())

        # Log metrics every epoch
        metrics = {
            "train/loss":  loss.item(),
            "val/loss":    val_loss,
            "val/accuracy": val_acc,
            "val/roc_auc":  val_auc,
            "epoch": epoch,
        }

        if WANDB_AVAILABLE:
            wandb.log(metrics, step=epoch)
        elif epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}: loss={loss.item():.4f}  val_acc={val_acc:.4f}  val_auc={val_auc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc

    if WANDB_AVAILABLE:
        wandb.summary["best_val_accuracy"] = best_val_acc
        run.finish()
        print(f"Run completed. Best val_accuracy: {best_val_acc:.4f}")
    else:
        print(f"[Simulated] Best val_accuracy: {best_val_acc:.4f} logged to wandb.summary")

    return best_val_acc


# Run a single experiment
config = {
    "hidden_dim":    128,
    "num_layers":    3,
    "dropout":       0.2,
    "learning_rate": 1e-3,
    "weight_decay":  1e-4,
    "epochs":        20,
}

print("Training with W&B logging...")
best_acc = train_with_wandb(config)

## Core Concept 2: wandb.log() — Logging Rich Data

W&B can log much more than just scalars. It supports:
- **Images**: `wandb.Image(img_array, caption="prediction")`
- **Tables**: `wandb.Table(data=rows, columns=["input", "pred", "label"])`
- **Histograms**: `wandb.Histogram(array)`
- **Plots**: matplotlib figures
- **Audio/Video**: `wandb.Audio()`, `wandb.Video()`
- **3D objects**: `wandb.Object3D()`

In [ ]:
# Demonstrate rich logging with W&B Tables and Images

def log_evaluation_results(model_name, y_true, y_pred, y_prob):
    """Log detailed evaluation results to W&B."""
    from sklearn.metrics import classification_report

    acc  = accuracy_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_prob)
    cm   = confusion_matrix(y_true, y_pred)

    if WANDB_AVAILABLE:
        run = wandb.init(project="evaluation-demo", name="eval_logging", mode="offline")

        # 1. Scalar summary
        wandb.log({"test/accuracy": acc, "test/roc_auc": auc})

        # 2. Confusion matrix as a W&B plot
        wandb.log({"confusion_matrix": wandb.plot.confusion_matrix(
            y_true=y_true.tolist(),
            preds=y_pred.tolist(),
            class_names=["Negative", "Positive"]
        )})

        # 3. ROC curve
        wandb.log({"roc_curve": wandb.plot.roc_curve(y_true, np.column_stack([1-y_prob, y_prob]))})

        # 4. Per-sample prediction table
        table = wandb.Table(columns=["sample_id", "true_label", "predicted", "probability", "correct"])
        for i in range(min(50, len(y_true))):
            table.add_data(i, int(y_true[i]), int(y_pred[i]), float(y_prob[i]), y_true[i] == y_pred[i])
        wandb.log({"predictions_table": table})

        # 5. Histogram of predicted probabilities
        wandb.log({"prob_histogram": wandb.Histogram(y_prob)})

        run.finish()
        print(f"Rich evaluation logged to W&B: acc={acc:.4f}, auc={auc:.4f}")
    else:
        print(f"[Simulated W&B rich logging] for {model_name}")
        print(f"  wandb.log({{'test/accuracy': {acc:.4f}, 'test/roc_auc': {auc:.4f}}})")
        print(f"  wandb.log({{'confusion_matrix': wandb.plot.confusion_matrix(...)}})") 
        print(f"  wandb.log({{'roc_curve': wandb.plot.roc_curve(...)}})") 
        print(f"  wandb.log({{'predictions_table': wandb.Table(...)}})")  # 50 rows
        print(f"  wandb.log({{'prob_histogram': wandb.Histogram(y_prob)}})")
        print(f"")
        print(f"  Results: acc={acc:.4f}, auc={auc:.4f}")
        print(f"  Confusion matrix: TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}")

# Run evaluation logging
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=50, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

log_evaluation_results("RandomForest", y_test, y_pred, y_prob)

## Core Concept 3: W&B Artifacts — Versioned Files

Artifacts version any file — datasets, models, reports.  
Think of it as Git for your data files.

```
dataset:v0 → dataset:v1 → dataset:v2  (lineage tracked)
    ↓              ↓
model:v0       model:v1  (know which data trained which model)
```

In [ ]:
import tempfile, os, pickle

if WANDB_AVAILABLE:
    run = wandb.init(project="artifacts-demo", name="artifact_example", mode="offline")

    # 1. Log a dataset artifact
    dataset_artifact = wandb.Artifact(
        name="classification-dataset",
        type="dataset",
        description="Synthetic classification dataset for MLP training",
        metadata={"n_samples": X.shape[0], "n_features": X.shape[1]}
    )
    # Save data to temp CSV and add to artifact
    with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False) as f:
        df = pd.DataFrame(X, columns=[f'f{i}' for i in range(X.shape[1])])
        df['label'] = y
        df.to_csv(f.name, index=False)
        dataset_artifact.add_file(f.name, name='data.csv')
        os.unlink(f.name)
    run.log_artifact(dataset_artifact)

    # 2. Log a model artifact
    rf_model = RandomForestClassifier(n_estimators=50, random_state=42)
    rf_model.fit(X_train, y_train)

    model_artifact = wandb.Artifact(
        name="rf-classifier",
        type="model",
        description="RandomForest classifier",
        metadata={"accuracy": float(rf_model.score(X_test, y_test))}
    )
    with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as f:
        pickle.dump(rf_model, f)
        model_artifact.add_file(f.name, name='model.pkl')
        os.unlink(f.name)
    run.log_artifact(model_artifact)

    run.finish()
    print("Dataset and model artifacts logged. Version tracked by W&B.")

    # Downloading an artifact (in a different run)
    print("\nTo use an artifact in another run:")
    print("  run = wandb.init(...)")
    print("  artifact = run.use_artifact('classification-dataset:latest')")
    print("  artifact_dir = artifact.download()  # returns local path")
    print("  df = pd.read_csv(os.path.join(artifact_dir, 'data.csv'))")
else:
    print("W&B Artifacts (simulated):")
    print()
    print("  # Creating a dataset artifact")
    print("  artifact = wandb.Artifact('my-dataset', type='dataset')")
    print("  artifact.add_file('data.csv')")
    print("  run.log_artifact(artifact)")
    print("  # → my-dataset:v0 stored, hash computed for deduplication")
    print()
    print("  # Later: update dataset")
    print("  artifact = wandb.Artifact('my-dataset', type='dataset')")
    print("  artifact.add_file('data_v2.csv')")
    print("  run.log_artifact(artifact)")
    print("  # → my-dataset:v1 created, full history preserved")
    print()
    print("  # Using an artifact in another run")
    print("  artifact = run.use_artifact('my-dataset:latest')")
    print("  path = artifact.download()  # fetches to ./artifacts/")
    print("  # W&B tracks lineage: run X used dataset:v1 to train model:v2")

## Core Concept 4: W&B Sweeps — Automated Hyperparameter Search

W&B Sweeps automatically tries many hyperparameter combinations.  
Strategies: **Random** (try random combos), **Grid** (try all), **Bayes** (smartly explore, exploit promising areas).

**Bayesian optimization**: after each trial, updates a model of which hyperparameters are promising.  
It is much more efficient than random search — typically finds good configs in fewer trials.

In [ ]:
# Define a sweep configuration
sweep_config = {
    "name": "mlp-hyperparameter-sweep",
    "method": "bayes",   # 'bayes', 'random', or 'grid'
    "metric": {          # what to optimize
        "name": "val/roc_auc",
        "goal": "maximize"
    },
    "parameters": {
        "hidden_dim": {
            "values": [64, 128, 256, 512]    # categorical: try each value
        },
        "num_layers": {
            "values": [2, 3, 4]
        },
        "learning_rate": {
            "distribution": "log_uniform_values",  # log scale search
            "min": 1e-4,
            "max": 1e-2
        },
        "dropout": {
            "distribution": "uniform",
            "min": 0.0,
            "max": 0.5
        },
        "weight_decay": {
            "distribution": "log_uniform_values",
            "min": 1e-6,
            "max": 1e-2
        },
        "epochs": {"value": 15}   # fixed — not a hyperparameter here
    },
    "early_terminate": {          # stop bad runs early (like Hyperband)
        "type": "hyperband",
        "s": 2,
        "eta": 3,
        "min_iter": 3
    }
}

import json
print("Sweep configuration:")
print(json.dumps(sweep_config, indent=2))

if WANDB_AVAILABLE:
    # Initialize sweep — returns a sweep_id
    sweep_id = wandb.sweep(sweep_config, project="mlp-classification")
    print(f"\nSweep ID: {sweep_id}")
    print("To start sweep agents: wandb agent <sweep_id>")
    print("Or run programmatically:")
    
    def sweep_train_fn():
        """Training function called by each sweep trial."""
        with wandb.init(mode="offline") as run:
            cfg = wandb.config
            acc = train_with_wandb(dict(cfg))
    
    # Run 3 sweep trials (normally you'd run 50-100)
    wandb.agent(sweep_id, function=sweep_train_fn, count=3)
else:
    print("\nSimulated sweep execution:")
    print()
    # Simulate what Bayesian search would do
    np.random.seed(0)
    print(f"{'Trial':>5}  {'hidden':>7}  {'layers':>6}  {'lr':>8}  {'dropout':>8}  {'val_auc':>8}")
    print("-" * 60)
    best = 0
    for trial in range(1, 11):
        hd  = np.random.choice([64, 128, 256, 512])
        nl  = np.random.choice([2, 3, 4])
        lr  = np.exp(np.random.uniform(np.log(1e-4), np.log(1e-2)))
        do  = np.random.uniform(0, 0.5)
        # Bayesian search would pick configs that maximize expected improvement
        auc = 0.82 + 0.12 * (hd/512) - 0.05 * do + 0.02 * np.random.randn()
        auc = min(0.97, max(0.75, auc))
        best = max(best, auc)
        print(f"{trial:>5}  {hd:>7}  {nl:>6}  {lr:>8.5f}  {do:>8.3f}  {auc:>8.4f}")
    print(f"\nBest val_auc: {best:.4f}")

## Core Concept 5: wandb.watch() — Tracking Gradients and Weights

`wandb.watch(model)` hooks into PyTorch to log gradient histograms and weight distributions every N steps.  
This helps diagnose training problems: vanishing gradients, dead neurons, exploding weights.

In [ ]:
print("wandb.watch() usage:")
print()
print("  import wandb")
print("  import torch.nn as nn")
print()
print("  run = wandb.init(project='my-project')")
print("  model = MyModel()")
print()
print("  # Watch model — logs gradients + weights")
print("  wandb.watch(")
print("      model,")
print("      log='all',          # 'gradients', 'parameters', 'all', or None")
print("      log_freq=100,       # log every 100 optimizer steps")
print("      log_graph=True,     # also log the model computation graph")
print("  )")
print()
print("  # Normal training loop — gradients logged automatically")
print("  for batch in dataloader:")
print("      loss = criterion(model(batch), targets)")
print("      loss.backward()")
print("      optimizer.step()")
print()
print("  # In the W&B dashboard, you'll see:")
print("  # - Histograms of each layer's gradients per step")
print("  # - Histograms of each layer's weights per step")
print("  # - Gradient norm curves (detect vanishing/exploding gradients)")
print()
print("Diagnostic signs to look for in the W&B dashboard:")
diag = [
    ("Gradient norms → 0",    "Vanishing gradients — reduce depth, use residuals or different activation"),
    ("Gradient norms → ∞",    "Exploding gradients — add gradient clipping"),
    ("Dead neurons (all 0 grad)", "ReLU dying — try LeakyReLU or better initialization"),
    ("Weight distribution unchanged", "Layers not learning — check learning rate, connectivity"),
]
for symptom, diagnosis in diag:
    print(f"  [{symptom}] → {diagnosis}")

## Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Forgetting `run.finish()` | Run stays "running" in dashboard | Use `with wandb.init():` context manager |
| Logging too frequently | Slow training, high W&B storage | Log every N steps, not every batch |
| Not using offline mode | Errors on machines without internet | Set `WANDB_MODE=offline` |
| API key exposed | Security risk | Never commit API key; use `wandb login` or env var |
| `wandb.config` not used in sweeps | Sweeps don't override hyperparams | Read `wandb.config` (not local dict) inside train function |
| Logging tensors instead of scalars | W&B can't display them | Call `.item()` on torch tensors before logging |
| Duplicate runs in Jupyter | Re-running cells creates new runs | Call `wandb.finish()` or check `wandb.run is not None` |

## Mini Project: Full Training Pipeline with W&B

In [ ]:
# Complete W&B-instrumented training pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

def full_ml_pipeline_with_wandb():
    """Complete ML pipeline with W&B tracking."""

    config = {
        "model": "GradientBoosting",
        "n_estimators": 150,
        "max_depth": 4,
        "learning_rate": 0.08,
        "subsample": 0.8,
        "dataset": "synthetic_3000",
        "n_cv_folds": 5,
    }

    if WANDB_AVAILABLE:
        run = wandb.init(
            project="mini-project",
            name="gb_final",
            config=config,
            tags=["gradient-boosting", "production-candidate"],
            notes="Final model candidate for production deployment"
        )

    # Cross-validation
    model = GradientBoostingClassifier(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        learning_rate=config["learning_rate"],
        subsample=config["subsample"],
        random_state=42
    )

    cv_scores = cross_val_score(model, X_train, y_train, cv=config["n_cv_folds"], scoring="roc_auc")

    if WANDB_AVAILABLE:
        for fold_i, score in enumerate(cv_scores):
            wandb.log({"cv_auc": score, "fold": fold_i + 1})
        wandb.log({"cv_auc_mean": cv_scores.mean(), "cv_auc_std": cv_scores.std()})

    # Final training on full train set
    model.fit(X_train, y_train)

    # Test evaluation
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    test_acc = accuracy_score(y_test, y_pred)
    test_auc = roc_auc_score(y_test, y_prob)
    cm = confusion_matrix(y_test, y_pred)

    if WANDB_AVAILABLE:
        wandb.log({
            "test/accuracy": test_acc,
            "test/roc_auc":  test_auc,
        })
        # Log confusion matrix
        wandb.log({"confusion_matrix": wandb.plot.confusion_matrix(
            y_true=y_test.tolist(), preds=y_pred.tolist(),
            class_names=["Negative", "Positive"]
        )})
        # Log feature importance
        feat_table = wandb.Table(
            columns=["feature", "importance"],
            data=[[f"feature_{i}", float(imp)] for i, imp in
                  sorted(enumerate(model.feature_importances_), key=lambda x: -x[1])[:10]]
        )
        wandb.log({"feature_importance": feat_table})

        # Save model as artifact
        with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as f:
            pickle.dump(model, f)
            artifact = wandb.Artifact("gb-model", type="model",
                                      metadata={"test_auc": test_auc})
            artifact.add_file(f.name, name="model.pkl")
            run.log_artifact(artifact)

        run.finish()

    print(f"Results:")
    print(f"  CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"  Test Accuracy: {test_acc:.4f}")
    print(f"  Test AUC: {test_auc:.4f}")
    print(f"  Confusion Matrix: TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}")
    if WANDB_AVAILABLE:
        print(f"  All results, plots, and model saved to W&B project 'mini-project'")

full_ml_pipeline_with_wandb()

## Interview Questions and Answers

In [ ]:
qa = [
    {
        "q": "What is the difference between W&B and MLflow?",
        "a": """
Both track ML experiments, but with different strengths:

W&B strengths:
- Rich visualizations: images, audio, video, 3D objects, custom charts
- Better team collaboration: shareable dashboards, Reports (like notebooks)
- Superior Sweeps: Bayesian optimization with early stopping built-in
- wandb.watch(): automatic gradient/weight monitoring
- Real-time streaming: see metrics live as training runs
- Primarily cloud-hosted (also has W&B Server for on-prem)

MLflow strengths:
- Fully open-source, self-hosted, free at any scale
- Model Registry: production-grade model lifecycle management
- Model serving built-in: mlflow models serve
- Multi-framework: sklearn, PyTorch, TF, XGBoost, LightGBM all supported
- Better for on-premises/compliance-restricted environments
- Deep Databricks integration

Many teams use BOTH: W&B for experiment tracking during development,
MLflow Model Registry for production deployment management.
        """
    },
    {
        "q": "How do W&B Sweeps work? Explain Bayesian optimization.",
        "a": """
W&B Sweeps coordinates hyperparameter search across runs:

1. You define a sweep config (search space + optimization method + metric to optimize)
2. W&B creates a sweep controller (a server that assigns configs to agents)
3. Sweep agents are processes that: get next config → train → report metric → repeat

Bayesian optimization (method='bayes'):
- Maintains a surrogate model (usually Gaussian Process or Tree Parzen Estimator)
- After each trial, updates the surrogate with (config, metric) data
- Uses Expected Improvement (EI) to choose the next config:
  EI = P(new config improves on best so far) × expected improvement amount
- Balances exploration (try unknown regions) vs exploitation (refine promising regions)

Why Bayesian > Random:
- Random: treats each trial independently, no learning from past trials
- Bayesian: each trial informs future trials, converges to good configs faster
- Typical speedup: 3-10× fewer trials to find equally good config

Early stopping with Hyperband:
- Runs all configs for 1 epoch, keeps top 1/eta
- Runs survivors for eta epochs, keeps top 1/eta again
- Repeats until 1 config remains — massive compute savings
        """
    },
    {
        "q": "What are W&B Artifacts and how do they enable reproducibility?",
        "a": """
W&B Artifacts are versioned, tracked files (datasets, models, reports) stored in W&B.

Each artifact version:
- Has a content hash (like a git commit hash) — identical files are deduplicated
- Tracks lineage: which run produced it, which runs consumed it
- Can be annotated with metadata (model accuracy, dataset statistics)

Reproducibility flow:
1. Log training data as an artifact: dataset:v3
2. Train model, log as artifact: model:v7 (produced from dataset:v3)
3. W&B shows the DAG: dataset:v3 → model:v7 → evaluation:v2

6 months later when you need to reproduce:
- 'Which data trained model:v7?' → W&B lineage shows dataset:v3
- Download exactly that data: artifact.download()
- Re-run training run that produced model:v7 (same code, same data)

Without artifacts: you'd search through S3 buckets hoping the files match.
With artifacts: the entire provenance chain is automatically recorded.
        """
    },
    {
        "q": "How would you use W&B in production (not just research)?",
        "a": """
In production, W&B is used for:

1. Model comparison before deployment:
   - Compare challenger vs champion models with side-by-side metric tables
   - Share reports with stakeholders showing why new model is better

2. Continuous training monitoring:
   - Scheduled retraining logs new runs to W&B automatically
   - Alert when model accuracy drops below threshold (W&B alerts via Slack/email)

3. Debugging production failures:
   - Log model predictions + inputs as W&B Tables during inference
   - When users report wrong predictions, find them in the Table and debug

4. Data drift detection:
   - Log input feature distributions in production as W&B histograms
   - Compare to training distributions — divergence indicates drift

5. A/B testing:
   - Log predictions from model A and model B separately
   - Compare business metrics (conversion rate, error rate) in real-time

Limitations:
- Not a serving platform (you still need FastAPI/BentoML/TorchServe)
- Cloud-hosted: data goes to W&B servers (W&B Server for on-premises)
- Cost: free tier limited; teams need paid plan
        """
    },
    {
        "q": "How do you handle W&B in environments without internet access?",
        "a": """
Several options:

1. Offline mode (most common):
   export WANDB_MODE=offline
   # OR in code:
   wandb.init(mode='offline')
   
   All data saved locally in ./wandb/ directory.
   Sync later when connected:
   wandb sync ./wandb/run-20240101_120000-abc123

2. W&B Server (self-hosted):
   - Install W&B on your own Kubernetes cluster
   - All data stays on-premises
   - Set MLFLOW_TRACKING_URI to your internal server

3. Disabled mode (CI/CD):
   wandb.init(mode='disabled')  # or WANDB_DISABLED=true
   # All wandb calls become no-ops — no errors, no logging

4. For unit tests:
   import wandb
   wandb.init(mode='disabled')
   # Your training code runs without W&B overhead in tests
        """
    },
]

for i, item in enumerate(qa, 1):
    print(f"Q{i}: {item['q']}")
    print(f"A: {item['a'].strip()}")
    print("-" * 70)
    print()

## Summary

| Feature | API | What It Does |
|---------|-----|-------------|
| Start run | `wandb.init(project=..., config=...)` | Creates a tracked experiment run |
| Log scalars | `wandb.log({'loss': 0.5, 'acc': 0.9})` | Records metrics per step |
| Log images | `wandb.log({'img': wandb.Image(arr)})` | Visual outputs |
| Log tables | `wandb.log({'preds': wandb.Table(...)})` | Per-sample prediction analysis |
| Watch model | `wandb.watch(model, log='all')` | Gradient/weight histograms |
| Artifacts | `wandb.Artifact(...)` | Version datasets and models |
| Sweeps | `wandb.sweep(config)` + `wandb.agent(id, fn)` | Automated HPO |
| Finish | `run.finish()` or context manager | End the run |

### Next Steps

1. **Free W&B account**: [https://wandb.ai/signup](https://wandb.ai/signup)
2. **Official quickstart**: [https://docs.wandb.ai/quickstart](https://docs.wandb.ai/quickstart)
3. **W&B courses (free)**: [https://www.wandb.courses/](https://www.wandb.courses/)
4. **Compare with MLflow**: Use W&B for tracking, MLflow for model registry in production
5. **Next**: Learn model serving frameworks (FastAPI, BentoML) to deploy your tracked models